## 1. Load the data
Loads the 300 synthetic customer notes (with their true PII values, our "ground truth") and the Enron emails, which we use to check for false alarms.

In [101]:
import os, re, sys
import pandas as pd

DATA_DIR = os.path.expanduser("~/bank-rag-guard/data")
SRC_DIR = os.path.expanduser("~/bank-rag-guard/src")
os.makedirs(SRC_DIR, exist_ok=True)           # folder for our reusable code

# dtype=str keeps phone numbers, IDs and IBANs as text instead of converting them to numbers
customers = pd.read_csv(f"{DATA_DIR}/customers.csv", dtype=str)   # includes the true emirates_id, iban, phone
enron = pd.read_json(f"{DATA_DIR}/enron_emails.jsonl", lines=True,
                     dtype=False, convert_dates=False)   # normal emails: should trigger few masks
print(len(customers), "customer notes;", len(enron), "emails")

300 customer notes; 1500 emails


## 2. Write the masker (`src/pii.py`)
`%%writefile` saves this cell's code as a reusable module. How it works:
- `normalize` converts Arabic-Indic digits (٠-٩) to ASCII digits one-for-one, so numbers written in Arabic still match and character positions stay the same.
- `PATTERNS` lists the identifiers it finds, most specific first. Once text is claimed by one pattern, later patterns skip it.
- **Payment cards** are found three ways: a number with a real card-brand prefix and length that passes the Luhn checksum; a 13-19 digit number with a card word (card, visa, mastercard, CVV and so on) nearby; or a standalone 15-16 digit number, which in a bank setting is almost always a card or account identifier.
- **Cue-word codes:** CVV, PIN, BIC and national ID numbers are masked when a word such as "CVV", "PIN", "BIC" or "SSN" appears just before them. Social security numbers are also recognised by layout: `123-45-6789`, `123 45 6789`, and the 756-prefixed Swiss format.
- **Network and web identifiers:** IP addresses (IPv4 and IPv6), MAC addresses, URLs and crypto addresses (Ethereum, Bitcoin, Litecoin).
- **Phone numbers** are recognised in five layouts: UAE mobiles, international numbers starting with `+`, local numbers with the area code in parentheses, local numbers starting with 0 and separated by space, hyphen or dot, and grouped numbers such as `713-853-1715` or `7422 257-1915`. Each has a digit-count check, so dates and ZIP codes are not mistaken for phones.
- **Account numbers** (8-16 digits) are masked when an account or payment word appears up to 60 characters before them, or when a transfer or payment word is followed by "to" and the number. Numbers directly after a currency symbol are treated as amounts and left alone, and 8-digit values that look like dates (YYYYMMDD) are skipped.
- IBAN checksums are deliberately not verified: for privacy masking it is safer to mask a mistyped IBAN than to leak it.
- `mask_text` replaces each finding with a tag such as `[IBAN]` and returns the list of findings.

In [102]:
%%writefile /home/linux1/bank-rag-guard/src/pii.py
"""Bilingual (English/Arabic) PII masker: structured identifiers."""
import re

# Map Arabic-Indic (٠-٩) and Persian (۰-۹) digits to 0-9. Both strings have 20 characters,
# so every character maps to exactly one character and text positions do not shift.
_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹", "01234567890123456789")

def normalize(text):
    """Return text with all non-ASCII digits replaced by ASCII digits (same length)."""
    return text.translate(_DIGITS)

def _digit_count(text):
    """How many digit characters a string contains."""
    return sum(ch.isdigit() for ch in text)

def _digits_between(low, high):
    """Build a check function: the match must contain between low and high digits."""
    return lambda text: low <= _digit_count(text) <= high

def _not_a_date(digits_text):
    """Reject 8-digit strings that look like a YYYYMMDD date (so timestamps are not masked)."""
    if len(digits_text) == 8:
        year, month, day = int(digits_text[:4]), int(digits_text[4:6]), int(digits_text[6:])
        if 1900 <= year <= 2100 and 1 <= month <= 12 and 1 <= day <= 31:
            return False
    return True

def _luhn_ok(text):
    """True if the digits in text pass the Luhn checksum used by payment cards."""
    numbers = [int(c) for c in text if c.isdigit()]
    total = 0
    for i, n in enumerate(reversed(numbers)):
        if i % 2 == 1:              # double every second digit from the right
            n *= 2
            if n > 9:
                n -= 9
        total += n
    return bool(numbers) and total % 10 == 0

def _brand_ok(text):
    """True if the digits start like a real card brand and have a length that brand uses."""
    d = "".join(c for c in text if c.isdigit())
    n, p2, p3, p4 = len(d), int(d[:2]), int(d[:3]), int(d[:4])
    if d[0] == "4":                                   return n in (16, 19)           # Visa
    if 51 <= p2 <= 55 or 2221 <= p4 <= 2720:          return n == 16                 # Mastercard
    if p2 in (34, 37):                                return n == 15                 # American Express
    if d[:4] == "6011" or p2 == 65 or 644 <= p3 <= 649: return n in (16, 19)         # Discover
    if 3528 <= p4 <= 3589:                            return 16 <= n <= 19           # JCB
    if p2 in (36, 38) or 300 <= p3 <= 305:            return n in (14, 16)           # Diners Club
    return False

def _not_a_timestamp(text):
    """Numbers starting with 19 or 20 are usually timestamps, and numbers with many leading zeros
    are usually record IDs, so neither is treated as a card."""
    return not text.startswith(("19", "20", "0000"))
    
def _looks_like_address(text):
    """Crypto addresses mix digits with lower- and upper-case letters; ordinary long words do not."""
    return (any(c.isdigit() for c in text) and any(c.islower() for c in text)
            and any(c.isupper() for c in text))

# Words that signal "an account or payment number follows" (English and Arabic)
_ACCOUNT_WORDS = r"(?:account|acct|acc't|a/c|حساب|الحساب|transfer(?:red)?|payment|deposit(?:ed)?|wire[d]?)"

# Words that signal "a payment card number is nearby"
_CARD_ALT = r"credit card|debit card|card|visa|mastercard|master card|maestro|amex|american express|discover|jcb|diners"

# Each entry: (label, compiled regex, regex group to mask (0 = whole match), optional check function).
# Order matters: specific patterns first, so later ones cannot claim their digits.
PATTERNS = [
    # Emirates ID: 784-YYYY-XXXXXXX-Z (hyphens or spaces optional)
    ("EMIRATES_ID", re.compile(r"(?<!\d)784[ \-]?\d{4}[ \-]?\d{7}[ \-]?\d(?!\d)"), 0, None),

    # IBAN, any country: 2 capital letters + 2 digits, then either a compact run of 11-30
    # letters/digits, or 3-7 groups of 4 separated by single spaces (plus an optional short last group).
    ("IBAN", re.compile(
        r"(?<![A-Za-z0-9])[A-Z]{2}\d{2}"
        r"(?:[A-Z0-9]{11,30}|(?: [A-Z0-9]{4}){3,7}(?: [A-Z0-9]{1,3})?)"
        r"(?![A-Za-z0-9])"), 0, lambda t: _digit_count(t) >= 8),

    # Payment card, way 1: real brand prefix and length, and a valid Luhn checksum
    ("CREDIT_CARD", re.compile(r"(?<!\d)\d(?:[ \-]?\d){12,18}(?!\d)"), 0,
     lambda t: _brand_ok(t) and _luhn_ok(t)),
    # Payment card, way 2a: a card word, then (up to 60 characters later) a 13-19 digit number
    ("CREDIT_CARD", re.compile(r"(?i)(?:" + _CARD_ALT + r")[^\d\n]{0,60}(?<!\d)(\d(?:[ \-]?\d){12,18})(?!\d)"),
     1, _digits_between(13, 19)),
    # Payment card, way 2b: a 13-19 digit number followed (within 40 characters) by a card word or CVV
    ("CREDIT_CARD", re.compile(r"(?<!\d)(\d(?:[ \-]?\d){12,18})(?!\d)(?=[^\d\n]{0,40}(?i:" + _CARD_ALT + r"|cvv))"),
     1, _digits_between(13, 19)),
    # Payment card, way 3: a standalone 15-16 digit number (not part of a longer token, not after a currency symbol)
    ("CREDIT_CARD", re.compile(r"(?<![\w\-./$€£¥,=])\d{15,16}(?![\w\-/]|\.\d)"), 0, _not_a_timestamp),
    
    # Crypto addresses: Ethereum (0x + 40 hex digits) and Bitcoin/Litecoin (base58, 25-39 characters)
    ("CRYPTO_ADDRESS", re.compile(r"(?<![A-Za-z0-9])0x[0-9a-fA-F]{40}(?![0-9a-fA-F])"), 0, None),
    ("CRYPTO_ADDRESS", re.compile(r"(?<![A-Za-z0-9])[13LM][1-9A-HJ-NP-Za-km-z]{24,38}(?![A-Za-z0-9])"),
     0, _looks_like_address),

    # IPv6: full form (8 groups), then compressed form with "::"
    ("IP_ADDRESS", re.compile(r"(?<![0-9A-Fa-f:])(?:[0-9A-Fa-f]{1,4}:){7}[0-9A-Fa-f]{1,4}(?![0-9A-Fa-f:])"), 0, None),
    ("IP_ADDRESS", re.compile(
        r"(?<![0-9A-Fa-f:])(?:[0-9A-Fa-f]{1,4}:){1,6}(?::[0-9A-Fa-f]{1,4}){1,6}(?![0-9A-Fa-f:])"), 0, None),

    # MAC address: six pairs of hex digits separated by ":" or "-"
    ("MAC_ADDRESS", re.compile(
        r"(?<![0-9A-Fa-f:\-])(?:[0-9A-Fa-f]{2}[:\-]){5}[0-9A-Fa-f]{2}(?![0-9A-Fa-f:\-])"), 0, None),

    # IPv4: four numbers from 0 to 255 separated by dots (a trailing sentence full stop is allowed)
    ("IP_ADDRESS", re.compile(
        r"(?<!\d)(?<!\d\.)(?:(?:25[0-5]|2[0-4]\d|1\d\d|[1-9]?\d)\.){3}(?:25[0-5]|2[0-4]\d|1\d\d|[1-9]?\d)(?!\d|\.\d)"),
     0, None),

    # US social security number, dashed: 123-45-6789
    ("SSN", re.compile(r"(?<![\d\-])\d{3}-\d{2}-\d{4}(?![\d\-])"), 0, None),
    # US social security number, spaced: 123 45 6789
    ("SSN", re.compile(r"(?<![\d\-.])\d{3} \d{2} \d{4}(?![\d\-.])"), 0, None),
    # Swiss social insurance layout: 756.XXXX.XXXX(.XX), with any separator or none (11 or 13 digits)
    ("SSN", re.compile(r"(?<![\d.])756[ .\-]?\d{4}[ .\-]?\d{4}(?:[ .\-]?\d{2})?(?!\d)"), 0, _digits_between(11, 13)),
    # National ID after a cue word: "SSN 123456789", "social security number 756.7496.6890"
    ("SSN", re.compile(r"(?i)(?:\bssn\b|social security(?: number)?|national (?:id|insurance)(?: number)?)"
                       r"[^\d\n]{0,25}(?<!\d)(\d(?:[ \-.]?\d){7,10})(?!\d)"), 1, _digits_between(8, 11)),

    # Card security code after a cue word: "CVV is 531", "security code being 383", "verification value (936)"
    ("CARD_CVV", re.compile(r"(?i)(?:\bcvv2?\b|\bcvc2?\b|security code|verification (?:code|value))"
                           r"[^\d\n]{0,25}?(?<!\d)(\d{3,4})(?!\d)"), 1, None),

    # PIN after the word PIN: "PIN - 8824", "PIN 1644"
    ("PIN", re.compile(r"(?i)(?<![A-Za-z])pin(?: code| number)?(?![A-Za-z])[^\d\n]{0,10}(?<!\d)(\d{4,6})(?!\d)"),
     1, None),

    # BIC / SWIFT code after the word BIC or SWIFT: 8 or 11 capital letters/digits in the BIC layout
    ("BIC", re.compile(r"(?i:\b(?:bic|swift)\b)[^\n]{0,30}?"
                       r"(?<![A-Za-z0-9])([A-Z]{6}[A-Z0-9]{2}(?:[A-Z0-9]{3})?)(?![A-Za-z0-9])"), 1, None),

    # UAE mobile: 05X XXX XXXX, 00971 5X XXX XXXX or +971 5X XXX XXXX
    ("PHONE", re.compile(r"(?<![\d\-])(?:(?:\+|00)971[ \-]?|0)5\d[ \-]?\d{3}[ \-]?\d{4}(?!\d)"), 0, None),

    # International phone: "+" country code, then 2-4 groups of 2-5 digits separated by space, hyphen or dot
    ("PHONE", re.compile(r"(?<![\w+])\+\d{1,4}(?:[ \-.]?\d{2,5}){2,4}(?!\d)"), 0, _digits_between(8, 15)),

    # Local phone, area code in parentheses: (98) 7561.5671 or (95)-5544-2455
    ("PHONE", re.compile(r"(?<!\w)\(\d{2,4}\)[ \-.]?\d{3,4}[ \-.]?\d{3,4}(?!\d)"), 0, _digits_between(8, 13)),

    # Local phone starting with 0: 03-4947-8651, 06.3603 7980, 0371-52754478, 07747 182426
    # (10-13 digits keeps out 9-digit ZIP+4 codes and short date fragments)
    ("PHONE", re.compile(r"(?<![\w\-.])0\d{1,4}[ \-.]\d{3,8}(?:[ \-.]\d{2,5})?(?!\d)"), 0, _digits_between(10, 13)),

    # Grouped phone without a leading 0 or +: 713-853-1715, 7422 257-1915, 1952-178-8156
    ("PHONE", re.compile(r"(?<![\w\-.])\d{3,4}[ \-.]\d{3}[ \-.]\d{4}(?!\d|[\-.]\d)"), 0, _digits_between(10, 11)),

    # Email address
    ("EMAIL", re.compile(r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}"), 0, None),

    # Web address starting with http(s):// or www. (trailing punctuation is not included)
    ("URL", re.compile(r'(?i)(?<![\w@])(?:https?://|www\.)[^\s<>"\')\]]*[^\s<>"\')\].,;:!?]'), 0, None),

    # Account number: an account/payment word, up to 60 non-digit characters, then 8-16 digits.
    # Group 1 is the number itself; a number right after a currency symbol is an amount and is skipped.
    ("ACCOUNT_NUMBER", re.compile(r"(?i)(?<!\w)" + _ACCOUNT_WORDS + r"[^\d\n]{0,60}(?<![\d$€£¥])(\d{8,16})(?!\d)"),
     1, _not_a_date),
    # Account number after a transfer or payment word and "to": "transfer of ¥2582... to 23564969"
    ("ACCOUNT_NUMBER", re.compile(r"(?i)(?<!\w)(?:transfer(?:red)?|wire[d]?|deposit(?:ed)?|payment|paid|sent?)\b"
                                  r"[^\n]{0,80}?\b(?:to|into)\s+(?:account\s+|acct\s+)?(?<![\d$€£¥])(\d{8,16})(?!\d)"),
     1, _not_a_date),
]

def find_pii(text):
    """Return a sorted list of (start, end, label) for every PII span found."""
    norm = normalize(text)                       # search the normalised text; positions also fit the original
    claimed, spans = [], []
    for label, pattern, group, check in PATTERNS:
        for m in pattern.finditer(norm):
            s, e = m.span(group)
            if s < 0:                            # the group did not take part in this match
                continue
            if check and not check(norm[s:e]):   # the pattern matched but failed its extra check
                continue
            if any(s < ce and e > cs for cs, ce in claimed):    # overlaps an earlier finding: skip
                continue
            claimed.append((s, e))
            spans.append((s, e, label))
    return sorted(spans)

def mask_text(text):
    """Return (masked_text, spans), with each finding replaced by a [LABEL] tag."""
    spans = find_pii(text)
    pieces, last = [], 0
    for s, e, label in spans:
        pieces.append(text[last:s])              # keep the text before the finding
        pieces.append(f"[{label}]")              # replace the finding with its tag
        last = e
    pieces.append(text[last:])                   # keep the rest
    return "".join(pieces), spans

Overwriting /home/linux1/bank-rag-guard/src/pii.py


## 3. Try the masker
Imports the module (reloading it if you edited it) and masks one English and one Arabic customer note. **Expected:** the Emirates ID, IBAN and phone number become tags. Names are still visible because name masking is part 2.

In [103]:
import importlib
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)             # lets Python find src/pii.py
import pii
importlib.reload(pii)                       # picks up edits without restarting the kernel

# --- Show one English and one Arabic example, before and after ---
for lang in ("en", "ar"):
    text = customers[customers["language"] == lang].iloc[0]["text"]
    masked, spans = pii.mask_text(text)
    print("BEFORE:", text)
    print("AFTER :", masked)
    print("FOUND :", [label for _, _, label in spans], "\n")

BEFORE: Customer Allison Hill, Emirates ID 784-1958-1719583-7, IBAN AE646921566149281915956, phone +971579254563, reported a disputed card transaction and requested a review.
AFTER : Customer Allison Hill, Emirates ID [EMIRATES_ID], IBAN [IBAN], phone [PHONE], reported a disputed card transaction and requested a review.
FOUND : ['EMIRATES_ID', 'IBAN', 'PHONE'] 

BEFORE: العميل محمود الألجاوي، رقم الهوية الإماراتية 784-2001-0109031-2، رقم الآيبان AE097772876166965439421، الهاتف +971593702683، قدّم شكوى بخصوص عملية بطاقة مشكوك فيها.
AFTER : العميل محمود الألجاوي، رقم الهوية الإماراتية [EMIRATES_ID]، رقم الآيبان [IBAN]، الهاتف [PHONE]، قدّم شكوى بخصوص عملية بطاقة مشكوك فيها.
FOUND : ['EMIRATES_ID', 'IBAN', 'PHONE'] 



## 3b. Display helper
`show` displays a table with a bold title. It can format chosen columns as percentages and draw in-cell bars for quick visual comparison. The cells below use it for all results.

In [104]:
from IPython.display import display, Markdown, HTML

def show(df, title, pct_cols=(), bars=None):
    """Display a DataFrame as a clean table. bars = {column: maximum value} draws in-cell bars."""
    display(Markdown(f"**{title}**"))
    # na_rep prints a dash for missing values (e.g. a recall that cannot be computed)
    styler = df.style.hide(axis="index").format({c: "{:.1%}" for c in pct_cols}, na_rep="–")
    for col, top in (bars or {}).items():
        styler = styler.bar(subset=[col], vmin=0, vmax=top, color="#5fa777")   # bar length = value / top
    display(styler)

## 4. Measure how well it works
- **Recall:** for each customer note, was the true Emirates ID, IBAN and phone number masked? Values are compared by their digits, so numbers written in Arabic digits are tested fairly. Results are shown overall and by field, language and digit style.
- **Tags on ordinary emails:** counts how often each tag fires across 1,500 Enron emails. UAE-specific tags (Emirates ID, IBAN, phone) should stay at or near zero. EMAIL firing is correct, because the emails contain real addresses.

In [105]:
def digits(s):
    """Digits only, after converting Arabic digits (used to compare values fairly)."""
    return re.sub(r"\D", "", pii.normalize(str(s)))      # str() guards against numeric values

# --- One row per (note, field): was the true value masked? ---
rows = []
for r in customers.itertuples():
    masked, _ = pii.mask_text(r.text)
    remaining = digits(masked)                            # tags contain no digits, so only unmasked digits remain
    for field in ("emirates_id", "iban", "phone"):
        rows.append({
            "Field": field,
            "Language": "Arabic" if r.language == "ar" else "English",
            "Digits": "Arabic digits" if re.search("[٠-٩]", r.text) else "ASCII digits",
            "Caught": digits(getattr(r, field)) not in remaining,
        })
res = pd.DataFrame(rows)

def recall_table(by):
    """Recall (share of values masked), hits and total, grouped by one column."""
    return res.groupby(by)["Caught"].agg(Recall="mean", Hits="sum", Total="count").reset_index()

display(Markdown(f"### Overall recall: {res['Caught'].mean():.1%} "
                 f"({res['Caught'].sum()} of {len(res)} values masked)"))
for by in ("Field", "Language", "Digits"):
    show(recall_table(by), f"Recall by {by.lower()}", pct_cols=["Recall"], bars={"Recall": 1})

# --- Which tags fire on ordinary emails ---
fired = {}
for text in enron["text"]:
    _, spans = pii.mask_text(text)
    for label in {l for _, _, l in spans}:               # count each tag once per email
        fired[label] = fired.get(label, 0) + 1

fa = pd.DataFrame(list(fired.items()), columns=["Tag", "Emails"])
fa["Share of emails"] = fa["Emails"] / len(enron)
fa["Note"] = fa["Tag"].map({"EMAIL": "Real addresses: correct catch",
                            "PHONE": "Real numbers in signatures are correct catches",
                            "URL": "Web addresses: correct catch, but very common in business email",
                            "CREDIT_CARD": "Brand-valid, card-cued or standalone 15-16 digit numbers: check the examples",
                            "IP_ADDRESS": "Addresses in mail headers: correct catches",
                            "ACCOUNT_NUMBER": "Needs an account or payment word nearby",
                            "CARD_CVV": "Needs a CVV word nearby; should be about zero",
                            "PIN": "Needs the word PIN nearby; should be about zero",
                            "BIC": "Needs the word BIC or SWIFT nearby; should be about zero",
                            "CRYPTO_ADDRESS": "Should be about zero",
                            "SSN": "Should be about zero",
                            "MAC_ADDRESS": "Should be about zero",
                            "IBAN": "Should be zero",
                            "EMIRATES_ID": "Should be zero"}).fillna("")
show(fa.sort_values("Emails", ascending=False),
     f"Tags firing on {len(enron):,} ordinary Enron emails",
     pct_cols=["Share of emails"], bars={"Share of emails": 1})

### Overall recall: 100.0% (900 of 900 values masked)

**Recall by field**

Field,Recall,Hits,Total
emirates_id,100.0%,300,300
iban,100.0%,300,300
phone,100.0%,300,300


**Recall by language**

Language,Recall,Hits,Total
Arabic,100.0%,456,456
English,100.0%,444,444


**Recall by digits**

Digits,Recall,Hits,Total
ASCII digits,100.0%,756,756
Arabic digits,100.0%,144,144


**Tags firing on 1,500 ordinary Enron emails**

Tag,Emails,Share of emails,Note
EMAIL,432,28.8%,Real addresses: correct catch
PHONE,411,27.4%,Real numbers in signatures are correct catches
URL,223,14.9%,"Web addresses: correct catch, but very common in business email"
IP_ADDRESS,14,0.9%,Addresses in mail headers: correct catches
CREDIT_CARD,6,0.4%,"Brand-valid, card-cued or standalone 15-16 digit numbers: check the examples"
ACCOUNT_NUMBER,3,0.2%,Needs an account or payment word nearby


## 5. Inspect what the masker caught
Shows up to five examples per tag from the Enron emails, with the masked part highlighted in yellow and 40 characters of context on either side. Use it to check that each match is a genuine piece of personal data.

In [106]:
import html

examples = {t: [] for t in ["CREDIT_CARD", "PHONE", "IP_ADDRESS", "SSN", "CARD_CVV", "PIN", "BIC", "ACCOUNT_NUMBER"]}
for text in enron["text"]:
    _, spans = pii.mask_text(text)
    for s, e, label in spans:
        if label in examples and len(examples[label]) < 5:
            before = text[max(0, s - 40):s].replace("\n", " ")
            after = text[e:e + 40].replace("\n", " ")
            # Card numbers are shown as their first four digits only; everything else in full
            shown = text[s:e][:4] + "…" if label == "CREDIT_CARD" else text[s:e]
            examples[label].append((before, shown, after))

body = ""
for label, items in examples.items():
    if not items:
        body += f"<tr><td><b>{label}</b></td><td><i>no matches</i></td></tr>"
    for before, match, after in items:
        body += ("<tr><td><b>" + label + "</b></td>"
                 "<td style='font-family:monospace;text-align:left'>…" + html.escape(before) +
                 "<mark style='background:#ffd54f;color:#000'>" + html.escape(match) + "</mark>" +
                 html.escape(after) + "…</td></tr>")
display(HTML("<table><tr><th>Tag</th><th style='text-align:left'>Match in context</th></tr>"
             + body + "</table>"))

Tag,Match in context
CREDIT_CARD,…TOTAL USD 137.50 PAYMENT: O/CCAX/+/CCCA5490…/1103/A011745 RESERVATION NUMBER(S) CO…
CREDIT_CARD,… INVOICE TOTAL USD 268.50 PAYMENT: CCCA5490…/1103/A011317 RESERVATION NUMBER(S) CO…
CREDIT_CARD,… INVOICE TOTAL USD 190.50 PAYMENT: CCAX3727…/1202/A000025 RESERVATION NUMBER(S) CO…
CREDIT_CARD,… INVOICE TOTAL USD 190.50 PAYMENT: CCVI4271…/1001/A259513 RESERVATION NUMBER(S) CO…
CREDIT_CARD,… INVOICE TOTAL USD 93.75 PAYMENT: CCAX3727…/1202/A000026 RESERVATION NUMBER(S) CO…
PHONE,"…rd Parkway Kansas City, MO Telephone: 816-756-1500 Guest Fax: 816-756-1635 Your reservat…"
PHONE,"…MO Telephone: 816-756-1500 Guest Fax: 816-756-1635 Your reservations are confirmed, see y…"
PHONE,…sas City! Sharon Brown ETS Gas Control 713-853-5660…
PHONE,…. slb Sharon Brown ETS Gas Control 713-853-5660…
PHONE,…ions. Sharon Brown ETS Gas Control 713-853-5660…


## 5b. Random samples of phone and card matches
Cell 5 shows only the first five matches, which can be unrepresentative. This cell draws 15 random phone matches and 8 random card matches from across all 1,500 emails (fixed seed, so the sample is repeatable), so we can judge how many are genuine. Card digits are hidden: only the first four digits and the length are shown.

In [107]:
import random, html

rng = random.Random(7)                               # fixed seed: the same sample every run
pool = {"PHONE": [], "CREDIT_CARD": []}

# --- Collect every match of the two tags, with context ---
for text in enron["text"]:
    for s, e, label in pii.find_pii(text):
        if label in pool:
            pool[label].append((text[max(0, s - 40):s].replace("\n", " "),
                                text[s:e],
                                text[e:e + 40].replace("\n", " ")))

# --- Draw random samples and render them ---
body = ""
for label, k in (("PHONE", 15), ("CREDIT_CARD", 8)):
    display(Markdown(f"**{label}: {len(pool[label])} matches in total; showing {min(k, len(pool[label]))} at random**"))
    for before, match, after in rng.sample(pool[label], min(k, len(pool[label]))):
        shown = f"{match[:4]}… ({len(match)} characters)" if label == "CREDIT_CARD" else match
        body += ("<tr><td><b>" + label + "</b></td>"
                 "<td style='font-family:monospace;text-align:left'>…" + html.escape(before) +
                 "<mark style='background:#ffd54f;color:#000'>" + html.escape(shown) + "</mark>" +
                 html.escape(after) + "…</td></tr>")
display(HTML("<table><tr><th>Tag</th><th style='text-align:left'>Match in context</th></tr>" + body + "</table>"))

**PHONE: 1058 matches in total; showing 15 at random**

**CREDIT_CARD: 6 matches in total; showing 6 at random**

Tag,Match in context
PHONE,"… Box 26530, Austin, Texas 78755 - Phone 512-795-8536 Kansas Game Set for 1 p.m., No TV; Bro…"
PHONE,…sion Support On-Call (713) 327-6032 [Pager]…
PHONE,"…on, please call the payroll hotline at 713.345.5555.…"
PHONE,"…n, DC 20036 Voice: 202.223.4700 Fax: 202.296.7138 **************************************…"
PHONE,"…tance, please contact Tracy Ramsey at 713-646-8311 or Jeff Leath at 713-646-6165. If yo…"
PHONE,…ion please call &Hoop-It-Up Hotline8: 713-521-4535 or E-Mail: Hoopitup@Texas.Net Join…
PHONE,…erez 713-345-708 Christian Smerz 713-345-7410 SITARA: No Scheduled Outages. S…
PHONE,…d to connect to the conference call is 712-271-3842. The passcode in which you will need t…
PHONE,… First Strategies!</a> Order Toll Free 800-538-7424 - 410-867-7424 All claims on third-par…
PHONE,…rnia Independent System Operator Voice: 916-351-2171 Fax: 916-351-2367 E-mail mneeley@cais…


## 6. Inspect the ai4privacy benchmark
Counts every label in the 1,000 benchmark examples and groups them by how our masker treats them:
1. **Masked now:** email, phone number, IBAN, account number
2. **Names (part 2):** first, last and middle names, and prefixes
3. **Add next:** payment cards, SSN, PIN, IP and MAC addresses, URLs, dates of birth and crypto addresses
4. **Other (not identifiers):** descriptive labels such as job area, gender or currency

The benchmark is English only, so Arabic is tested on our own synthetic data. The grouping is our own scoping decision.

In [108]:
import json
from collections import Counter

bench = pd.read_json(f"{DATA_DIR}/pii_benchmark.jsonl", lines=True,
                     dtype=False, convert_dates=False)

def get_mask(value):
    """privacy_mask is a list of {value, label, start, end}; it may be stored as a JSON string."""
    return json.loads(value) if isinstance(value, str) else value

counts = Counter(m["label"] for mask in bench["privacy_mask"] for m in get_mask(mask))

# --- Our scoping: which benchmark labels belong to which stage ---
MASKED_NOW = {"EMAIL", "PHONENUMBER", "IBAN", "ACCOUNTNUMBER"}
NAMES = {"FIRSTNAME", "LASTNAME", "MIDDLENAME", "PREFIX"}
ADD_NEXT = {"CREDITCARDNUMBER", "CREDITCARDCVV", "SSN", "PIN", "BIC", "IPV4", "IPV6", "IP",
            "MAC", "URL", "DOB", "BITCOINADDRESS", "ETHEREUMADDRESS", "LITECOINADDRESS"}

def group_of(label):
    if label in MASKED_NOW: return "1. Masked now"
    if label in NAMES:      return "2. Names (part 2)"
    if label in ADD_NEXT:   return "3. Add next"
    return "4. Other (not identifiers)"

table = pd.DataFrame(list(counts.items()), columns=["Label", "Count"])
table["Group"] = table["Label"].map(group_of)

display(Markdown(f"**{len(bench):,} examples, languages: {bench['language'].value_counts().to_dict()}** "
                 f"(English only, so Arabic is tested on our own synthetic data)"))

summary = table.groupby("Group")["Count"].agg(Labels="count", Spans="sum").reset_index()
summary["Share of all spans"] = summary["Spans"] / summary["Spans"].sum()
show(summary, "Benchmark labels by what our masker does with them",
     pct_cols=["Share of all spans"], bars={"Share of all spans": 1})

show(table.sort_values(["Group", "Count"], ascending=[True, False]),
     f"All {len(table)} labels", bars={"Count": int(table["Count"].max())})

# --- One example ---
row = bench.iloc[0]
display(Markdown(f"**Example text:** {row['source_text']}"))
show(pd.DataFrame(get_mask(row["privacy_mask"]))[["label", "value"]], "Its labelled PII")

**1,000 examples, languages: {'en': 1000}** (English only, so Arabic is tested on our own synthetic data)

**Benchmark labels by what our masker does with them**

Group,Labels,Spans,Share of all spans
1. Masked now,4,246,7.9%
2. Names (part 2),4,541,17.5%
3. Add next,14,569,18.4%
4. Other (not identifiers),34,1740,56.2%


**All 56 labels**

Label,Count,Group
EMAIL,84,1. Masked now
PHONENUMBER,67,1. Masked now
ACCOUNTNUMBER,64,1. Masked now
IBAN,31,1. Masked now
FIRSTNAME,299,2. Names (part 2)
LASTNAME,99,2. Names (part 2)
PREFIX,79,2. Names (part 2)
MIDDLENAME,64,2. Names (part 2)
URL,72,3. Add next
CREDITCARDNUMBER,63,3. Add next


**Example text:** A student's assessment was found on device bearing IMEI: 06-184755-866851-3. The document falls under the various topics discussed in our Optimization curriculum. Can you please collect it?

**Its labelled PII**

label,value
PHONEIMEI,06-184755-866851-3
JOBAREA,Optimization


## 7. Score the masker on the benchmark
`score_benchmark` runs the masker on every benchmark example and compares its findings with the labelled PII:
- **Recall:** the share of labelled values (of the four types we cover) that our masker tagged with the matching tag. A match means our span covers at least half of the labelled one.
- **Precision (same label):** the share of our findings that overlap a labelled value of the matching type.
- **Precision (any PII label):** the share of our findings that overlap a labelled value of any type. It is more forgiving, because the benchmark may label the same text differently.

The function also returns the missed values and the findings that matched nothing, so we can see why scores are low.

In [109]:
# --- Map benchmark label names to our tag names ---
# MASKEDNUMBER is scored as CREDIT_CARD: in this data every MASKEDNUMBER value is a 16-digit card-like
# number (see cell 8d). This is a scoping decision, so we state it in the report.
LABEL_MAP = {"EMAIL": "EMAIL", "PHONENUMBER": "PHONE", "IBAN": "IBAN", "ACCOUNTNUMBER": "ACCOUNT_NUMBER",
             "CREDITCARDNUMBER": "CREDIT_CARD", "MASKEDNUMBER": "CREDIT_CARD", "CREDITCARDCVV": "CARD_CVV",
             "SSN": "SSN", "PIN": "PIN", "BIC": "BIC",
             "IPV4": "IP_ADDRESS", "IPV6": "IP_ADDRESS", "IP": "IP_ADDRESS",
             "MAC": "MAC_ADDRESS", "URL": "URL",
             "BITCOINADDRESS": "CRYPTO_ADDRESS", "LITECOINADDRESS": "CRYPTO_ADDRESS",
             "ETHEREUMADDRESS": "CRYPTO_ADDRESS"}
TAGS = ["EMAIL", "PHONE", "IBAN", "ACCOUNT_NUMBER", "CREDIT_CARD", "CARD_CVV", "SSN", "PIN", "BIC",
        "IP_ADDRESS", "MAC_ADDRESS", "URL", "CRYPTO_ADDRESS", "EMIRATES_ID"]   # EMIRATES_ID has no benchmark labels

def overlap(a_start, a_end, b_start, b_end):
    """Number of characters shared by two spans (0 if they do not touch)."""
    return max(0, min(a_end, b_end) - max(a_start, b_start))

def score_benchmark(bench_df):
    """Return (scores table, missed values per tag, unmatched findings per tag)."""
    st = {t: dict(gold=0, caught=0, pred=0, same=0, anyl=0) for t in TAGS}
    missed = {t: [] for t in TAGS}
    unmatched = {t: [] for t in TAGS}

    for row in bench_df.itertuples():
        text = row.source_text
        gold = get_mask(row.privacy_mask)             # labelled PII: value, label, start, end
        preds = pii.find_pii(text)                    # our findings: (start, end, tag)

        # --- Recall side: was each labelled value masked with the right tag? ---
        for g in gold:
            tag = LABEL_MAP.get(g["label"])
            if tag is None:
                continue                              # a label type we do not handle yet
            st[tag]["gold"] += 1
            half = 0.5 * (g["end"] - g["start"])
            if any(p[2] == tag and overlap(p[0], p[1], g["start"], g["end"]) >= half for p in preds):
                st[tag]["caught"] += 1
            else:
                missed[tag].append(g["value"])

        # --- Precision side: does each finding overlap something that was labelled? ---
        for s, e, tag in preds:
            st[tag]["pred"] += 1
            same = any(LABEL_MAP.get(g["label"]) == tag and overlap(s, e, g["start"], g["end"]) > 0 for g in gold)
            anyl = any(overlap(s, e, g["start"], g["end"]) > 0 for g in gold)
            st[tag]["same"] += same
            st[tag]["anyl"] += anyl
            if not anyl:
                unmatched[tag].append(text[s:e])

    rows = []
    for t in TAGS:
        d = st[t]
        rows.append({"Tag": t,
                     "Labelled": d["gold"],
                     "Recall": d["caught"] / d["gold"] if d["gold"] else None,
                     "Found": d["pred"],
                     "Precision (same label)": d["same"] / d["pred"] if d["pred"] else None,
                     "Precision (any PII label)": d["anyl"] / d["pred"] if d["pred"] else None})
    return pd.DataFrame(rows), missed, unmatched

scores, missed, unmatched = score_benchmark(bench)
show(scores, "Masker scores on the ai4privacy benchmark (1,000 English examples)",
     pct_cols=["Recall", "Precision (same label)", "Precision (any PII label)"],
     bars={"Recall": 1, "Precision (same label)": 1, "Precision (any PII label)": 1})

**Masker scores on the ai4privacy benchmark (1,000 English examples)**

Tag,Labelled,Recall,Found,Precision (same label),Precision (any PII label)
EMAIL,84,100.0%,84,100.0%,100.0%
PHONE,67,100.0%,67,100.0%,100.0%
IBAN,31,100.0%,31,100.0%,100.0%
ACCOUNT_NUMBER,64,71.9%,47,97.9%,100.0%
CREDIT_CARD,127,96.9%,125,98.4%,100.0%
CARD_CVV,21,66.7%,14,100.0%,100.0%
SSN,33,100.0%,33,100.0%,100.0%
PIN,17,29.4%,5,100.0%,100.0%
BIC,19,42.1%,8,100.0%,100.0%
IP_ADDRESS,146,100.0%,146,100.0%,100.0%


## 8. Inspect misses and wrong findings
For each tag, shows up to five labelled values the masker **missed** and up to five findings that matched **no labelled PII**. Misses show which formats the patterns do not cover yet. Unmatched findings are either wrong matches or real PII the benchmark did not label.

In [110]:
rows = []
for tag in [t for t in TAGS if t != "EMIRATES_ID"]:
    for value in missed[tag][:5]:
        rows.append({"Tag": tag, "Kind": "Missed", "Text": value})
    for value in unmatched[tag][:5]:
        rows.append({"Tag": tag, "Kind": "Matched nothing", "Text": value})

show(pd.DataFrame(rows), "Examples of misses and unmatched findings")

**Examples of misses and unmatched findings**

Tag,Kind,Text
ACCOUNT_NUMBER,Missed,19713884
ACCOUNT_NUMBER,Missed,06189047
ACCOUNT_NUMBER,Missed,81410291
ACCOUNT_NUMBER,Missed,92445827
ACCOUNT_NUMBER,Missed,78871431
CREDIT_CARD,Missed,8833835460537634
CREDIT_CARD,Missed,6152796726792205
CREDIT_CARD,Missed,2582384409779685
CREDIT_CARD,Missed,0518878019678390
CARD_CVV,Missed,597


## 8b. Context around missed values
Shows up to eight labelled phone numbers and eight labelled account numbers that the masker missed, with 50 characters of surrounding text and the missed value highlighted. The surrounding words show which phrasings a new rule would need to cover.

In [111]:
import html

def miss_contexts(bench_df, label, n=8, width=50):
    """Return up to n (text before, missed value, text after) tuples for one benchmark label."""
    tag = LABEL_MAP[label]
    found = []
    for row in bench_df.itertuples():
        text = row.source_text
        preds = pii.find_pii(text)
        for g in get_mask(row.privacy_mask):
            if g["label"] != label:
                continue
            half = 0.5 * (g["end"] - g["start"])
            caught = any(p[2] == tag and overlap(p[0], p[1], g["start"], g["end"]) >= half for p in preds)
            if not caught:
                found.append((text[max(0, g["start"] - width):g["start"]],
                              text[g["start"]:g["end"]],
                              text[g["end"]:g["end"] + width]))
            if len(found) >= n:
                return found
    return found

body = ""
for label in ("PHONENUMBER", "ACCOUNTNUMBER", "SSN", "CREDITCARDNUMBER"):
    items = miss_contexts(bench, label)
    if not items:
        body += f"<tr><td><b>{label}</b></td><td><i>no misses</i></td></tr>"
    for before, value, after in items:
        body += ("<tr><td><b>" + label + "</b></td>"
                 "<td style='font-family:monospace;text-align:left'>…" + html.escape(before) +
                 "<mark style='background:#ffd54f;color:#000'>" + html.escape(value) + "</mark>" +
                 html.escape(after) + "…</td></tr>")
display(HTML("<table><tr><th>Label</th><th style='text-align:left'>Missed value in context</th></tr>"
             + body + "</table>"))

Label,Missed value in context
PHONENUMBER,no misses
ACCOUNTNUMBER,…tient's medical procedure has been requested. The 19713884 needs to be verified first though.…
ACCOUNTNUMBER,…Our compliance department noticed 06189047 under the name of Mr. Ward has unusual transactio…
ACCOUNTNUMBER,…ant next month. You can book an appointment using 81410291 or transfer funds to SM72L00155880065622T8777237.…
ACCOUNTNUMBER,"…An important disclosure, patient 92445827 adverse reaction is irreversible. Pause trial for…"
ACCOUNTNUMBER,…adverse reaction is irreversible. Pause trial for 78871431.…
ACCOUNTNUMBER,…lend it to us by filling the teacher survey using 63363202 as your ID.…
ACCOUNTNUMBER,…Psychometric tests for subject code 46778430 are ready. Visit https://used-coke.name/ to view …
ACCOUNTNUMBER,…s involved in a legal issue. Property falls under 94266990.…
SSN,no misses


## 8c. Sample values for the next label types
Shows up to four distinct example values for each benchmark label we plan to add patterns for (cards, SSN, IP and MAC addresses, URLs, dates of birth, crypto addresses and more). The patterns are written to fit these real formats.

In [112]:
NEXT_LABELS = ["CREDITCARDNUMBER", "CREDITCARDCVV", "SSN", "PIN", "BIC", "IPV4", "IPV6", "IP", "MAC",
               "URL", "DOB", "BITCOINADDRESS", "ETHEREUMADDRESS", "LITECOINADDRESS", "MASKEDNUMBER"]

samples = {}
for row in bench.itertuples():
    for g in get_mask(row.privacy_mask):
        if g["label"] in NEXT_LABELS:
            bucket = samples.setdefault(g["label"], [])
            if len(bucket) < 4 and g["value"] not in bucket:    # up to four distinct examples
                bucket.append(g["value"])

table_next = pd.DataFrame([{"Label": k, "Example values": "   |   ".join(v)} for k, v in samples.items()])
show(table_next, "Example values for the next label types")

**Example values for the next label types**

Label,Example values
MASKEDNUMBER,5890724654311332 | 2191941526453503 | 5982215161172252 | 0760991685235721
DOB,"29/12/1957 | 16/11/1983 | February 4, 1909 | 11/43"
IPV6,edaf:fd8f:e1e8:cfec:8bab:1afd:6aad:550c | 4d36:f8ad:4ebd:1bff:c5eb:62aa:b4d7:dddf | 41bd:5dec:8ee3:c6c0:026a:fa3f:8b18:febc | 6b3b:c6fc:209a:e5b1:9b4f:ea5b:8a5e:e58e
PIN,8824 | 3790 | 1644 | 4896
IPV4,109.4.61.213 | 215.114.180.213 | 250.59.196.86 | 152.58.179.27
CREDITCARDNUMBER,2948447876023997 | 6381973478101820 | 6259724424541296 | 1417506813461516
BITCOINADDRESS,34EUu2QzxVkmBLo4anvhitwWwV1ux6vDW | 1wcy8u4zbKiZA4CANutQohXaN49EJmFioXJZ | 34nfuko7AiX7MXkz6syvQ38auUycszmMwMQEw | 3u5rCmT62cBj4qj9dcfH4FGzgvjaD6
LITECOINADDRESS,3CkiYCgdy1gmYJPwHjU2xjuLNtM7T | LPYNSnbUXUxn2N3uCKTvrfYRFMadFuiQB | MXbUiawvqmRuZaqZNmH28GAhL8FwinWLB | 37qfqkykZPVGZmcBzFA9xCQVLJXHNS
MAC,08:7a:81:2f:48:fd | db:c0:c1:05:bf:5d | 62:d3:0a:3a:0d:1b | 39:70:a3:03:a8:bd
SSN,479-13-8850 | 153-10-3105 | 280-41-6336 | 75674966890


## 8d. Context for cue-dependent labels
For cards, CVV codes, PINs, dates of birth, BIC codes, masked numbers and SSNs, shows up to five labelled values with 45 characters of context on each side. These labels only make sense with surrounding words, so rules must be written from real phrasing. It also reports how many benchmark card numbers pass the Luhn checksum, which tells us whether the checksum is a fair test on this data.

In [113]:
import html

CUE_LABELS = ["CREDITCARDNUMBER", "CREDITCARDCVV", "PIN", "DOB", "BIC", "MASKEDNUMBER", "SSN"]

# --- Luhn pass rate for the benchmark's card numbers ---
cards = [g["value"] for row in bench.itertuples() for g in get_mask(row.privacy_mask)
         if g["label"] == "CREDITCARDNUMBER"]
passed = sum(pii._luhn_ok(c) for c in cards)
display(Markdown(f"**Benchmark card numbers passing the Luhn checksum:** {passed} of {len(cards)} "
                 f"({passed / max(len(cards), 1):.0%})"))

# --- Collect up to five contexts per label ---
counts = {l: 0 for l in CUE_LABELS}
body = ""
for row in bench.itertuples():
    text = row.source_text
    for g in get_mask(row.privacy_mask):
        label = g["label"]
        if label in counts and counts[label] < 5:
            counts[label] += 1
            before = text[max(0, g["start"] - 45):g["start"]]
            after = text[g["end"]:g["end"] + 45]
            body += ("<tr><td><b>" + label + "</b></td>"
                     "<td style='font-family:monospace;text-align:left'>…" + html.escape(before) +
                     "<mark style='background:#ffd54f;color:#000'>" + html.escape(g["value"]) + "</mark>" +
                     html.escape(after) + "…</td></tr>")
display(HTML("<table><tr><th>Label</th><th style='text-align:left'>Value in context</th></tr>"
             + body + "</table>"))

**Benchmark card numbers passing the Luhn checksum:** 11 of 63 (17%)

Label,Value in context
MASKEDNUMBER,…ergency supplies in 16356 need a refill. Use 5890724654311332 to pay for them.…
DOB,…Your recent hospital data recorded on 29/12/1957 regarding chronic disease management has bee…
PIN,"…sonal discussions, ensure privacy with PIN - 8824 and EYECOLOR - Eye color: Brown.…"
MASKEDNUMBER,…patient Leuschke. We've been assigned number 2191941526453503 for identification.…
CREDITCARDNUMBER,…ards issued by discover bearing numbers like 2948447876023997.…
CREDITCARDNUMBER,…activities related to the credit card number 6381973478101820. The transactions are coming from the IP add…
DOB,"…Appointment reminder for Rebeca on 16/11/1983, paid via mastercard number ending in 598221…"
MASKEDNUMBER,"…1/1983, paid via mastercard number ending in 5982215161172252.…"
DOB,"…gned to accommodate diet needs based on your February 4, 1909.…"
MASKEDNUMBER,…d that the last four digits of your card are 0760991685235721 on our end. Please contact our office if thi…


## 9. Save the scores
Appends this run's scores to `eval/pii_benchmark_scores.csv` under a run name. A run name can only be used once, so earlier runs are never overwritten by accident. Later runs are added to the same file, so the final report can show every stage.

In [115]:
EVAL_DIR = os.path.expanduser("~/bank-rag-guard/eval")
os.makedirs(EVAL_DIR, exist_ok=True)

def save_scores(df, run, overwrite=False):
    """Append a run's scores to the CSV. Refuses to reuse a run name unless overwrite=True."""
    path = f"{EVAL_DIR}/pii_benchmark_scores.csv"
    out = df.assign(Run=run)
    if os.path.exists(path):
        old = pd.read_csv(path)
        if run in set(old["Run"]) and not overwrite:
            raise ValueError(f"A run named '{run}' already exists. Choose a new name.")
        out = pd.concat([old[old["Run"] != run], out], ignore_index=True)
    out.to_csv(path, index=False)
    print("Saved", len(df), "rows for run:", run)

save_scores(scores, "final rules (dev set)")

Saved 14 rows for run: final rules (dev set)


## 10. Compare runs
Reads every saved run from `eval/pii_benchmark_scores.csv` and shows recall and precision per tag, one column per run, so improvements are easy to see. This table is the basis for the results section of the report.

In [116]:
history = pd.read_csv(f"{EVAL_DIR}/pii_benchmark_scores.csv")
tags_we_score = [t for t in TAGS if t != "EMIRATES_ID"]
runs = list(dict.fromkeys(history["Run"]))               # runs in the order they were saved

for metric in ("Recall", "Precision (same label)", "Precision (any PII label)"):
    wide = (history.pivot(index="Tag", columns="Run", values=metric)
                   .reindex(tags_we_score)[runs].reset_index())
    show(wide, f"{metric} by run", pct_cols=runs, bars={r: 1 for r in runs})

**Recall by run**

Tag,baseline (UAE-focused patterns),international phone and IBAN patterns,card brands and cue words,"wider phone, ID and number rules",held-out test (final rules),final rules (dev set)
EMAIL,100.0%,100.0%,100.0%,100.0%,100.0%,100.0%
PHONE,1.5%,37.3%,91.0%,100.0%,96.2%,100.0%
IBAN,0.0%,100.0%,100.0%,100.0%,100.0%,100.0%
ACCOUNT_NUMBER,64.1%,64.1%,68.8%,71.9%,78.8%,71.9%
CREDIT_CARD,–,–,45.7%,96.9%,98.1%,96.9%
CARD_CVV,–,–,66.7%,66.7%,66.7%,66.7%
SSN,–,–,69.7%,100.0%,95.9%,100.0%
PIN,–,–,29.4%,29.4%,33.3%,29.4%
BIC,–,–,42.1%,42.1%,66.7%,42.1%
IP_ADDRESS,–,–,100.0%,100.0%,100.0%,100.0%


**Precision (same label) by run**

Tag,baseline (UAE-focused patterns),international phone and IBAN patterns,card brands and cue words,"wider phone, ID and number rules",held-out test (final rules),final rules (dev set)
EMAIL,100.0%,100.0%,100.0%,100.0%,100.0%,100.0%
PHONE,100.0%,100.0%,100.0%,100.0%,100.0%,100.0%
IBAN,–,100.0%,100.0%,100.0%,97.7%,100.0%
ACCOUNT_NUMBER,82.0%,85.4%,78.6%,97.9%,100.0%,97.9%
CREDIT_CARD,–,–,96.7%,98.4%,98.1%,98.4%
CARD_CVV,–,–,100.0%,100.0%,100.0%,100.0%
SSN,–,–,100.0%,100.0%,100.0%,100.0%
PIN,–,–,100.0%,100.0%,100.0%,100.0%
BIC,–,–,100.0%,100.0%,100.0%,100.0%
IP_ADDRESS,–,–,100.0%,100.0%,100.0%,100.0%


**Precision (any PII label) by run**

Tag,baseline (UAE-focused patterns),international phone and IBAN patterns,card brands and cue words,"wider phone, ID and number rules",held-out test (final rules),final rules (dev set)
EMAIL,100.0%,100.0%,100.0%,100.0%,100.0%,100.0%
PHONE,100.0%,100.0%,100.0%,100.0%,100.0%,100.0%
IBAN,–,100.0%,100.0%,100.0%,100.0%,100.0%
ACCOUNT_NUMBER,100.0%,100.0%,100.0%,100.0%,100.0%,100.0%
CREDIT_CARD,–,–,100.0%,100.0%,100.0%,100.0%
CARD_CVV,–,–,100.0%,100.0%,100.0%,100.0%
SSN,–,–,100.0%,100.0%,100.0%,100.0%
PIN,–,–,100.0%,100.0%,100.0%,100.0%
BIC,–,–,100.0%,100.0%,100.0%,100.0%
IP_ADDRESS,–,–,100.0%,100.0%,100.0%,100.0%


## 11. Held-out test on unseen benchmark examples
The patterns were written after studying the first 1,000 benchmark examples, so scores on them are optimistic. This cell downloads 1,000 different examples from far later in the same dataset (starting at row 20,000), keeps the English ones, and scores the masker **unchanged**. Do not tune rules on this set; it is only useful while it stays unseen. The gap between the two score tables is an honest estimate of how well the rules generalise.

In [118]:
import requests, time

HF = "https://datasets-server.huggingface.co"
DATASET = "ai4privacy/pii-masking-200k"

def hf_rows_range(dataset, split, start, n):
    """Return n rows starting at row `start`, read through Hugging Face's web API (100 per request)."""
    info = requests.get(f"{HF}/splits", params={"dataset": dataset}, timeout=30)
    info.raise_for_status()
    config = next(s["config"] for s in info.json()["splits"] if s["split"] == split)
    rows = []
    while len(rows) < n:
        length = min(100, n - len(rows))
        for attempt in range(5):                     # retry briefly if the service is busy
            r = requests.get(f"{HF}/rows", timeout=60, params={
                "dataset": dataset, "config": config, "split": split,
                "offset": start + len(rows), "length": length})
            if r.status_code == 200:
                break
            time.sleep(3 * (attempt + 1))
        r.raise_for_status()
        batch = [item["row"] for item in r.json()["rows"]]
        if not batch:
            break
        rows.extend(batch)
    return rows

held = pd.DataFrame(hf_rows_range(DATASET, "train", 20000, 1000))
print("Languages in this slice:", held["language"].value_counts().to_dict())
held = held[held["language"] == "en"].reset_index(drop=True)     # our masker is scored on English only
print(len(held), "English examples")

# --- Score the masker exactly as it is: no changes allowed after seeing these numbers ---
held_scores, _, _ = score_benchmark(held)
rate_cols = ["Recall", "Precision (same label)", "Precision (any PII label)"]
show(held_scores, f"Held-out scores ({len(held)} unseen English examples)",
     pct_cols=rate_cols, bars={c: 1 for c in rate_cols})
# overwrite=True is correct here: the held-out row should always match the current final rules
save_scores(held_scores, "held-out test (final rules)", overwrite=True)

Languages in this slice: {'en': 1000}
1000 English examples


**Held-out scores (1000 unseen English examples)**

Tag,Labelled,Recall,Found,Precision (same label),Precision (any PII label)
EMAIL,94,100.0%,94,100.0%,100.0%
PHONE,52,96.2%,50,100.0%,100.0%
IBAN,42,100.0%,43,97.7%,100.0%
ACCOUNT_NUMBER,80,78.8%,63,100.0%,100.0%
CREDIT_CARD,106,98.1%,106,98.1%,100.0%
CARD_CVV,27,66.7%,18,100.0%,100.0%
SSN,49,95.9%,47,100.0%,100.0%
PIN,18,33.3%,6,100.0%,100.0%
BIC,30,66.7%,20,100.0%,100.0%
IP_ADDRESS,168,100.0%,168,100.0%,100.0%


Saved 14 rows for run: held-out test (final rules)
